# Pandas for Data Engineering

While navigating through this notebook keep a Data Engineering mindset: correctness, consistency, and reproducibility.


## 1) Setup: imports, logging, and project folders

In Data Engineering work, having a predictable folder structure makes your pipeline easier to debug and rerun safely.

We will use:
- `data/raw/`   : files as received (no guarantees)
- `data/bronze/`: minimally parsed + quarantine outputs
- `data/silver/`: cleaned and validated outputs
- `data/output/`: consumer-facing reports

Dependencies:
- `pandas`, `numpy`
- Parquet support requires one engine:
  - Recommended: `pip install pyarrow`
  - Alternative: `pip install fastparquet`


In [ ]:

import json
import logging
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Tuple

import numpy as np
import pandas as pd

logging.basicConfig(level=logging.INFO, format="%(levelname)s | %(message)s")
logger = logging.getLogger("pandas-de")

BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
OUTPUT_DIR = DATA_DIR / "output"

for d in [RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

(RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR)


## 2) Pandas basics in a pipeline context: Series and DataFrame

Pandas provides two primary data structures:
- **Series**: one-dimensional labeled array
- **DataFrame**: two-dimensional labeled table (similar to a SQL table)

In Data Engineering, a DataFrame usually represents a batch of records to clean, validate, and publish.

Key inspection tools you should default to:
- `head()` to sample rows
- `shape` for row/column count
- `info()` for dtypes and null counts
- `describe(include="all")` for quick stats


In [ ]:
# Series example
s1 = pd.Series([3, 5, 1, 2, 7])
s1


In [ ]:
# DataFrame example
df_example = pd.DataFrame({"x": [1, 2, 3], "y": ["a", "b", "c"]})
df_example


## 3) Build a consistent dataset theme: people master + events log

We will create two small datasets and then write them to disk.

People master (`people_df`):
- stable key: `person_id`
- descriptive attributes: `name`, `city`, `gender`
- semi-business fields: `salary_usd`, `signup_date`

Events log (`events_df`):
- stable key: `event_id`
- references people via `person_id`
- schema drift: extra fields (e.g., `campaign_id`, `reason`)
- missing field case: one event missing `event_type`
- unknown key case: one event references an unknown `person_id` (to demonstrate joins)


In [ ]:

people_records = [
    {"person_id": 1001, "name": "Alice",   "age": 29,  "gender": "F", "city": "Oxfordshire",   "email": "alice@example.com",   "signup_date": "2024-01-15"},
    {"person_id": 1002, "name": "Bob",     "age": 83,  "gender": "M", "city": "Marshall",      "email": None,                  "signup_date": "2023-11-03"},
    {"person_id": 1003, "name": "Charlie", "age": 34,  "gender": "M", "city": "Kansas City",   "email": "charlie@example.com", "signup_date": "2024-02-10"},
    {"person_id": 1004, "name": "Bastian", "age": 12,  "gender": "M", "city": "De Forest",     "email": "bastian@example.com", "signup_date": "2024-07-01"},
    {"person_id": 1005, "name": "Ella",    "age": 79,  "gender": "F", "city": "Newport News",  "email": "ella@example.com",    "signup_date": "2022-06-20"},
    {"person_id": 1006, "name": "Jaco",    "age": 35,  "gender": "M", "city": "Norristown",    "email": "jaco@example.com",    "signup_date": "2024-04-05"},
    {"person_id": 1007, "name": "Ash",     "age": None,"gender": "M", "city": "Pallet Town",   "email": "ash@example.com",     "signup_date": "2024-05-18"},
    {"person_id": 1010, "name": "Alice",   "age": 30,  "gender": "F", "city": "Oxfordshire",   "email": "alice2@example.com",  "signup_date": "2025-01-10"},
]

people_df = pd.DataFrame(people_records)
people_df["salary_usd"] = [85000, np.nan, 95000, None, 72000, 110000, 65000, 90000]

events_records = [
    {"event_id": "e-0001", "person_id": 1001, "event_type": "login",    "event_ts": "2025-02-01T08:10:00", "source": "web"},
    {"event_id": "e-0002", "person_id": 1001, "event_type": "purchase", "event_ts": "2025-02-01T08:22:00", "source": "web", "amount_usd": 19.99},
    {"event_id": "e-0003", "person_id": 1002, "event_type": "login",    "event_ts": "2025-02-03T12:00:00", "source": "mobile"},
    {"event_id": "e-0004", "person_id": 9999, "event_type": "login",    "event_ts": "2025-02-03T12:05:00", "source": "web"},  # unknown person_id
    {"event_id": "e-0005", "person_id": 1003, "event_type": "purchase", "event_ts": "2025-02-05T17:45:00", "source": "mobile", "amount_usd": 120.00},
    {"event_id": "e-0006", "person_id": 1005, "event_type": "refund",   "event_ts": "2025-02-06T09:20:00", "source": "web", "amount_usd": 120.00, "reason": "duplicate_charge"},
    {"event_id": "e-0007", "person_id": 1006, "event_type": "login",    "event_ts": "2025-02-06T10:00:00", "source": "web", "campaign_id": "cmp-55"},
    {"event_id": "e-0008", "person_id": 1007,                 "event_ts": "2025-02-07T11:30:00", "source": "mobile"},  # missing event_type
]

events_df = pd.DataFrame(events_records)

people_df.head(), events_df.head()


## 4) File I/O

A common beginner pitfall is referencing files that don't exist. Here we create them.

We will write:
- `data/raw/people.csv`
- `data/raw/events.jsonl`

We also add one malformed JSON line at the end of the JSONL file to simulate a real ingestion failure.


In [ ]:

people_csv_path = RAW_DIR / "people.csv"
events_jsonl_path = RAW_DIR / "events.jsonl"

people_df.to_csv(people_csv_path, index=False)

with events_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in events_records:
        f.write(json.dumps(rec) + "\n")
    # Malformed JSON line (missing closing brace)
    f.write('{"event_id":"e-9999","person_id":1001,"event_type":"login","event_ts":"2025-02-08T08:00:00"\n')

people_csv_path, events_jsonl_path


### Exercise 4.1 (TODO): Read people.csv and inspect schema

Task:
- Read `people_csv_path` into `raw_people_df`
- Create `people_columns` as a list of column names
- Create `people_dtypes` as a dict mapping column -> dtype string


In [ ]:
# TODO
raw_people_df = None
people_columns = None
people_dtypes = None


## 5) Indexing and selection: boolean masks, `.loc`, `.iloc`

Most pipeline selection work should use:
- a boolean mask
- `.loc[mask, columns]`

This keeps code explicit and avoids chained indexing bugs.


In [ ]:
people_work_df = raw_people_df.copy()
people_work_df.head()


In [ ]:
adult_mask = pd.to_numeric(people_work_df["age"], errors="coerce").fillna(-1) >= 18
adults_df = people_work_df.loc[adult_mask, ["person_id","name","age","city"]]
adults_df


### Exercise 5.1 (TODO): Filter high-salary people with `.loc`

Task:
- Create `high_salary_df` where `salary_usd >= 90000`
- Keep columns: `person_id`, `name`, `salary_usd`
- Use a boolean mask + `.loc`


In [ ]:
# TODO
high_salary_df = None


### Exercise 5.1 (Checks)


In [ ]:
assert list(high_salary_df.columns) == ["person_id","name","salary_usd"]
assert (pd.to_numeric(high_salary_df["salary_usd"], errors="coerce") >= 90000).all()


## 6) Cleaning: types, missing values, duplicates

Cleaning is about making data consistent and safe to use.

We will:
1) normalize types with coercion (`to_numeric`, `to_datetime`)
2) handle missing values intentionally
3) deduplicate by key (not by whole row)


In [ ]:
filtered_df = people_work_df[people_work_df["age"].fillna(-1) > 30]
dropped_df = people_work_df.drop(columns=["city"])
renamed_df = people_work_df.rename(columns={"age": "years"})
(filtered_df.head(), dropped_df.head(), renamed_df.head())


In [ ]:
clean_people_df = people_work_df.copy()
clean_people_df["signup_date"] = pd.to_datetime(clean_people_df["signup_date"], errors="coerce")
clean_people_df["age"] = pd.to_numeric(clean_people_df["age"], errors="coerce")
clean_people_df["salary_usd"] = pd.to_numeric(clean_people_df["salary_usd"], errors="coerce")
clean_people_df["email"] = clean_people_df["email"].fillna("unknown@example.com")
clean_people_df["salary_usd"] = clean_people_df["salary_usd"].fillna(0.0)
clean_people_df.isna().mean().sort_values(ascending=False)


### Exercise 6.1 (TODO): Fill missing age using the median


In [ ]:
# TODO
age_filled_people_df = None


### Exercise 6.1 (Checks)


In [ ]:
assert pd.api.types.is_numeric_dtype(age_filled_people_df["age"])
assert age_filled_people_df["age"].isna().sum() == 0


In [ ]:
dup_row = age_filled_people_df.loc[age_filled_people_df["person_id"] == 1003].copy()
people_with_dup = pd.concat([age_filled_people_df, dup_row], ignore_index=True)
people_with_dup[people_with_dup["person_id"] == 1003]


### Exercise 6.2 (TODO): Deduplicate by `person_id`


In [ ]:
# TODO
people_deduped_df = None


### Exercise 6.2 (Checks)


In [ ]:
assert people_deduped_df.duplicated(subset=["person_id"]).sum() == 0
assert people_deduped_df["person_id"].nunique() == people_deduped_df.shape[0]


## 7) Data contracts with `dataclass` (production-like validation)

We validate `people` with explicit checks that raise `ValueError` with clear messages.


In [ ]:

@dataclass(frozen=True)
class PeopleContract:
    required_columns: Tuple[str, ...] = ("person_id","name","age","gender","city","email","signup_date","salary_usd")
    key_column: str = "person_id"
    gender_allowed: Tuple[str, ...] = ("F","M")
    age_min: float = 0.0
    age_max: float = 120.0
    salary_min: float = 0.0

    def validate(self, df: pd.DataFrame) -> None:
        missing = set(self.required_columns) - set(df.columns)
        if missing:
            raise ValueError(f"Missing required columns: {sorted(missing)}")

        if df[self.key_column].isna().any():
            raise ValueError(f"{self.key_column} must not be null")

        if not df[self.key_column].is_unique:
            raise ValueError(f"{self.key_column} must be unique")

        if not pd.api.types.is_numeric_dtype(df["age"]):
            raise ValueError("age must be numeric")

        if not df["age"].between(self.age_min, self.age_max).all():
            raise ValueError("age contains values outside allowed range")

        if not set(df["gender"].dropna().unique()).issubset(set(self.gender_allowed)):
            raise ValueError("gender contains unexpected values")

        if not pd.api.types.is_datetime64_any_dtype(df["signup_date"]):
            raise ValueError("signup_date must be datetime")

        if not pd.api.types.is_numeric_dtype(df["salary_usd"]):
            raise ValueError("salary_usd must be numeric")

        if (df["salary_usd"] < self.salary_min).any():
            raise ValueError("salary_usd must be non-negative")

people_contract = PeopleContract()


In [ ]:
people_validated_df = people_deduped_df.copy()
people_validated_df["signup_date"] = pd.to_datetime(people_validated_df["signup_date"], errors="coerce")
people_validated_df["age"] = pd.to_numeric(people_validated_df["age"], errors="coerce")
people_validated_df["salary_usd"] = pd.to_numeric(people_validated_df["salary_usd"], errors="coerce").fillna(0.0)

people_contract.validate(people_validated_df)
logger.info("PeopleContract validation: OK")

people_validated_df.head()


## 8) Robust ingestion of JSON Lines with quarantine


In [ ]:
def read_jsonl_with_quarantine(path: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    valid: List[Dict[str, Any]] = []
    bad: List[Dict[str, Any]] = []
    with path.open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            raw = line.rstrip("\n")
            if not raw.strip():
                continue
            try:
                valid.append(json.loads(raw))
            except Exception as e:
                bad.append({"line_no": line_no, "error": str(e), "raw_line": raw})
    return pd.DataFrame(valid), pd.DataFrame(bad)

events_ok_df, events_bad_df = read_jsonl_with_quarantine(events_jsonl_path)
logger.info(f"Events parsed: ok={events_ok_df.shape[0]} bad={events_bad_df.shape[0]}")

quarantine_path = BRONZE_DIR / "events_quarantine.csv"
events_bad_df.to_csv(quarantine_path, index=False)

events_ok_df.head(), events_bad_df


### Exercise 8.1 (TODO): Standardize events


In [ ]:
# TODO
events_validated_df = None


### Exercise 8.1 (Checks)


In [ ]:

assert isinstance(events_validated_df, pd.DataFrame)
assert {"event_id","person_id","event_ts","event_type"}.issubset(set(events_validated_df.columns))
assert events_validated_df["event_id"].notna().all()
assert events_validated_df["person_id"].notna().all()
assert events_validated_df["event_ts"].notna().all()
assert events_validated_df["event_type"].isna().sum() == 0
assert pd.api.types.is_datetime64_any_dtype(events_validated_df["event_ts"])


## 9) Writing curated outputs (silver) and idempotency


In [ ]:
people_silver_path = SILVER_DIR / "people.parquet"
events_silver_path = SILVER_DIR / "events.parquet"

people_validated_df.to_parquet(people_silver_path, index=False)
events_validated_df.to_parquet(events_silver_path, index=False)

people_silver_df = pd.read_parquet(people_silver_path)
events_silver_df = pd.read_parquet(events_silver_path)

people_silver_df.shape, events_silver_df.shape


### Exercise 9.1 (TODO): Round-trip row counts


In [ ]:
# TODO
people_row_count = None
events_row_count = None


### Exercise 9.1 (Checks)


In [ ]:
assert isinstance(people_row_count, int) and people_row_count > 0
assert isinstance(events_row_count, int) and events_row_count > 0
assert people_row_count == people_silver_df.shape[0]
assert events_row_count == events_silver_df.shape[0]


## 10) Joining and aggregations: enrich events and compute metrics


In [ ]:
enriched_events_df = events_silver_df.merge(
    people_silver_df[["person_id","city","gender"]],
    on="person_id",
    how="left",
    validate="many_to_one",
)
assert enriched_events_df.shape[0] == events_silver_df.shape[0]
unmatched = enriched_events_df["city"].isna().sum()
logger.info(f"Unmatched events (unknown person_id): {unmatched}")
enriched_events_df.head()


In [ ]:
tmp = enriched_events_df.copy()
tmp["city"] = tmp["city"].fillna("UNKNOWN")

event_counts = (
    tmp.groupby(["city","event_type"], as_index=False)
       .agg(event_count=("event_id","count"))
       .sort_values(["city","event_type"])
)
event_counts


### Exercise 10.1 (TODO): Aggregation totals


In [ ]:
# TODO
total_events_from_counts = None
total_events_raw = None


### Exercise 10.1 (Checks)


In [ ]:
assert isinstance(total_events_from_counts, int)
assert isinstance(total_events_raw, int)
assert total_events_from_counts == total_events_raw


In [ ]:

pivot_report = event_counts.pivot_table(
    index="city",
    columns="event_type",
    values="event_count",
    aggfunc="sum",
    fill_value=0,
).reset_index()
pivot_report


### Exercise 10.2 (TODO): Melt pivot report


In [ ]:
# TODO
tidy_again_df = None


### Exercise 10.2 (Checks)


In [ ]:
assert set(tidy_again_df.columns) == {"city","event_type","event_count"}
assert tidy_again_df.shape[0] == pivot_report.shape[0] * (pivot_report.shape[1] - 1)


## 11) Quick EDA: descriptive statistics


In [ ]:
people_silver_df.describe(include="all").T


In [ ]:
null_rates = people_silver_df.isna().mean().sort_values(ascending=False)
null_rates


## 12) Large datasets: chunking pattern (runnable)


In [ ]:
large_csv_path = RAW_DIR / "people_large.csv"
large_df = pd.concat([people_silver_df] * 3000, ignore_index=True)
large_df.to_csv(large_csv_path, index=False)
large_csv_path, large_df.shape


In [ ]:
def process_chunk(chunk: pd.DataFrame) -> Dict[str, int]:
    placeholder_emails = int((chunk["email"] == "unknown@example.com").sum())
    missing_age = int(chunk["age"].isna().sum())
    return {"rows": int(len(chunk)), "placeholder_emails": placeholder_emails, "missing_age": missing_age}

chunk_stats: List[Dict[str, int]] = []
for chunk in pd.read_csv(large_csv_path, chunksize=10_000):
    chunk_stats.append(process_chunk(chunk))

chunk_stats[:3], len(chunk_stats)


### Exercise 12.1 (TODO): Rows processed


In [ ]:
# TODO
rows_processed = None


### Exercise 12.1 (Checks)


In [ ]:
assert isinstance(rows_processed, int)
assert rows_processed == large_df.shape[0]


## 13) Mini-lab


In [ ]:

def run_pipeline(raw_dir: Path, bronze_dir: Path, silver_dir: Path, output_dir: Path) -> Dict[str, Path]:
    people = pd.read_csv(raw_dir / "people.csv")
    people["signup_date"] = pd.to_datetime(people["signup_date"], errors="coerce")
    people["age"] = pd.to_numeric(people["age"], errors="coerce")
    people["salary_usd"] = pd.to_numeric(people["salary_usd"], errors="coerce").fillna(0.0)
    people["email"] = people["email"].fillna("unknown@example.com")
    people["age"] = people["age"].fillna(people["age"].median())
    people = people.drop_duplicates(subset=["person_id"], keep="last")
    people_contract.validate(people)

    events_ok, events_bad = read_jsonl_with_quarantine(raw_dir / "events.jsonl")
    quarantine_out = bronze_dir / "events_quarantine_capstone.csv"
    events_bad.to_csv(quarantine_out, index=False)

    events_ok["event_ts"] = pd.to_datetime(events_ok["event_ts"], errors="coerce")
    if "event_type" not in events_ok.columns:
        events_ok["event_type"] = "unknown"
    else:
        events_ok["event_type"] = events_ok["event_type"].fillna("unknown")

    if events_ok["event_ts"].isna().any():
        raise ValueError("Some events could not be parsed as datetime. Check raw inputs.")
    if events_ok["event_id"].isna().any() or events_ok["person_id"].isna().any():
        raise ValueError("event_id and person_id must not be null")

    enriched = events_ok.merge(
        people[["person_id","city","gender"]],
        on="person_id",
        how="left",
        validate="many_to_one",
    )
    if enriched.shape[0] != events_ok.shape[0]:
        raise ValueError("Join changed row count unexpectedly")

    tmp = enriched.copy()
    tmp["city"] = tmp["city"].fillna("UNKNOWN")
    report = (
        tmp.groupby(["city","event_type"], as_index=False)
           .agg(event_count=("event_id","count"))
           .sort_values(["city","event_type"])
    )

    people_out = silver_dir / "people_capstone.parquet"
    events_out = silver_dir / "events_capstone.parquet"
    report_out = output_dir / "report_capstone.csv"

    people.to_parquet(people_out, index=False)
    events_ok.to_parquet(events_out, index=False)
    report.to_csv(report_out, index=False)

    return {"people_parquet": people_out, "events_parquet": events_out, "quarantine_csv": quarantine_out, "report_csv": report_out}

capstone_artifacts = run_pipeline(RAW_DIR, BRONZE_DIR, SILVER_DIR, OUTPUT_DIR)
capstone_artifacts


### Exercise 13.1 (TODO): Verify capstone outputs


In [ ]:
# TODO
report_total = None
events_total = None


### Exercise 13.1 (Checks)


In [ ]:
report_df = pd.read_csv(capstone_artifacts["report_csv"])
events_capstone_df = pd.read_parquet(capstone_artifacts["events_parquet"])

assert list(report_df.columns) == ["city","event_type","event_count"]
assert report_total == int(report_df["event_count"].sum())
assert events_total == int(events_capstone_df.shape[0])
assert report_total == events_total


> Content created by [**Carlos Cruz-Maldonado**](https://www.linkedin.com/in/carloscruzmaldonado/).  
> I am available to answer any questions or provide further assistance.   
> Feel free to reach out to me at any time.  